In [1]:
import os 
import json 
import joblib 
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [2]:
os.makedirs('models',exist_ok=True)
os.makedirs('reports',exist_ok=True)

fake = pd.read_csv("C:/Users/zaid/Downloads/fake_new_project/Fake_news_detector_dataset/fake.csv")
real = pd.read_csv("C:/Users/zaid/Downloads/fake_new_project/Fake_news_detector_dataset/True.csv")

fake["label"] = 0
real["label"] = 1
def prepare_text(df):
    title = df["title"].fillna("") if "title" in df else ""
    body = df["text"].fillna("") if "text" in df else ""

    if isinstance(title, str):
        return body

    return (title + " " + body).str.strip()

fake["content"] = prepare_text(fake)
real["content"] = prepare_text(real)

data = pd.concat([fake, real], ignore_index=True)
data = data[["content", "label"]]

data["content"] = data["content"].astype(str).str.strip()
data = data[data["content"].str.len() > 30]
data = data.drop_duplicates(subset=["content"])

# 3. Split dataset
X = data["content"]
y = data["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 4. Define models
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        C=1.0
    ),
    "Naive Bayes": MultinomialNB()
}

results = []
trained_models = {}

# 5. Train and evaluate
for name, classifier in models.items():

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            stop_words="english",
            max_features=50000,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )),
        ("classifier", classifier)
    ])

    print(f"Training {name}...")

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions),
        "recall": recall_score(y_test, predictions),
        "f1_score": f1_score(y_test, predictions)
    }

    results.append(metrics)
    trained_models[name] = pipeline

# 6. Compare models
comparison = pd.DataFrame(results)
comparison.to_csv("reports/model_comparison.csv", index=False)

best_name = comparison.sort_values(
    "f1_score", ascending=False
).iloc[0]["model"]

best_model = trained_models[best_name]

# 7. Final evaluation
predictions = best_model.predict(X_test)

final_metrics = {
    "model_name": best_name,
    "accuracy": float(accuracy_score(y_test, predictions)),
    "precision": float(precision_score(y_test, predictions)),
    "recall": float(recall_score(y_test, predictions)),
    "f1_score": float(f1_score(y_test, predictions)),
    "training_samples": int(len(X_train)),
    "testing_samples": int(len(X_test)),
    "classification_report": classification_report(
        y_test,
        predictions,
        target_names=["Fake", "Real"],
        output_dict=True
    )
}

# 8. Save model
joblib.dump(best_model, "models/fake_news_pipeline.pkl")

with open("reports/metrics.json", "w") as file:
    json.dump(final_metrics, file, indent=4)

# 9. Confusion matrix
cm = confusion_matrix(y_test, predictions)

plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Fake", "Real"],
    yticklabels=["Fake", "Real"]
)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title(f"Confusion Matrix - {best_name}")

plt.tight_layout()
plt.savefig("reports/confusion_matrix.png")
plt.close()

print("\nTraining completed!")
print("Best model:", best_name)
print("Accuracy:", final_metrics["accuracy"])
print("F1 Score:", final_metrics["f1_score"])
print("Saved model successfully!")


Training Logistic Regression...
Training Naive Bayes...

Training completed!
Best model: Logistic Regression
Accuracy: 0.9891318245748626
F1 Score: 0.9900129244507109
Saved model successfully!


#  Use this code for open in streamlit browser
# >> python -m streamlit run app.py<<